# Chapitre 8 — Observabilité du RAG

[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ahouahounko/rag-en-pratique/blob/main/chapters/chapitre-08-observabilite/08_observabilite.ipynb)

Ce notebook couvre les 10 exemples du chapitre. OpenAI est facultatif ; l'attribution, la latence et la dérive s'exécutent localement.

## Ressources utiles

- [OpenAI Docs — observabilité et usage](https://developers.openai.com/api/docs/guides/agents-api/observability)
- [OpenAI Docs — tracing](https://developers.openai.com/api/docs/guides/agents-api/tracing)
- [RAGAS — métriques](https://docs.ragas.io/en/stable/concepts/metrics/available_metrics/)
- [SciPy — test KS](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.ks_2samp.html)

## 0. Préparer Colab ou Jupyter

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

if not Path("src").is_dir():
    if not Path("rag-en-pratique").is_dir():
        subprocess.run(["git", "clone", "https://github.com/Ahouahounko/rag-en-pratique.git"], check=True)
    os.chdir("rag-en-pratique")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", ".[observability]"],
    check=True,
)
print("Environnement du chapitre 8 prêt :", Path.cwd())


## Configuration OpenAI facultative

In [ ]:
# @title Activer OpenAI pour les exemples 1, 2, 3 et 5
UTILISER_OPENAI = False # @param {type:"boolean"}

if UTILISER_OPENAI:
    import os
    import subprocess
    import sys
    from getpass import getpass

    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[openai]"], check=True)
    if not os.getenv("OPENAI_API_KEY"):
        os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY : ")
    if not os.getenv("OPENAI_MODEL"):
        os.environ["OPENAI_MODEL"] = input("OPENAI_MODEL : ").strip()
    print("OpenAI est activé pour la génération et les juges.")
else:
    print("OpenAI désactivé. Les analyses locales restent exécutables.")


## 1. Jeu de référence

Équilibrer cas directs, synthèses et abstentions attendues.

Script correspondant : [`01_generation_jeu.py`](examples/01_generation_jeu.py)

In [ ]:
# ruff: noqa: F811
"""Générer un jeu de référence équilibré à partir d'un corpus."""

from __future__ import annotations

import itertools
import random
from collections import defaultdict

from rag_en_pratique.observability import PassageEvaluation
from rag_en_pratique.prompting import generer, openai_configure


def _tirer(elements: list, nombre: int, rng: random.Random) -> list:
    if not elements or nombre <= 0:
        return []
    return rng.sample(elements, min(nombre, len(elements)))


def generer_jeu(
    passages: list[PassageEvaluation],
    taille: int,
    repartition: dict[str, float],
    sujets_voisins: list[str],
    *,
    client=None,
    model: str | None = None,
    graine: int = 42,
) -> list[dict[str, object]]:
    """Crée des questions directes, de synthèse et sans réponse."""
    if taille <= 0:
        raise ValueError("taille doit être positive")
    types = ("directe", "synthese", "sans_reponse")
    if set(repartition) != set(types) or abs(sum(repartition.values()) - 1.0) > 1e-9:
        raise ValueError(f"La répartition doit contenir {types} et totaliser 1")
    rng = random.Random(graine)
    comptes = {nom: int(taille * repartition[nom]) for nom in types}
    comptes["directe"] += taille - sum(comptes.values())
    jeu: list[dict[str, object]] = []

    for passage in _tirer(passages, comptes["directe"], rng):
        question = generer(
            "Écris une question réaliste dont la réponse est entièrement dans le passage.",
            passage.texte,
            client=client,
            model=model,
        )
        reponse = generer(
            "Réponds uniquement avec le passage fourni.",
            f"PASSAGE : {passage.texte}\nQUESTION : {question}",
            client=client,
            model=model,
        )
        jeu.append(
            {
                "question": question,
                "reponse": reponse,
                "passages_attendus": [passage.id],
                "type": "directe",
            }
        )

    par_source: dict[str, list[PassageEvaluation]] = defaultdict(list)
    for passage in passages:
        par_source[passage.source].append(passage)
    couples = [
        couple
        for groupe in par_source.values()
        for couple in itertools.combinations(groupe, 2)
    ]
    for premier, second in _tirer(couples, comptes["synthese"], rng):
        question = generer(
            "Écris une question réaliste qui exige les deux passages pour répondre complètement.",
            f"PASSAGE 1 : {premier.texte}\nPASSAGE 2 : {second.texte}",
            client=client,
            model=model,
        )
        reponse = generer(
            "Réponds uniquement avec les deux passages.",
            f"PASSAGES : {premier.texte}\n{second.texte}\nQUESTION : {question}",
            client=client,
            model=model,
        )
        jeu.append(
            {
                "question": question,
                "reponse": reponse,
                "passages_attendus": [premier.id, second.id],
                "type": "synthese",
            }
        )

    for sujet in _tirer(sujets_voisins, comptes["sans_reponse"], rng):
        question = generer(
            "Écris une question plausible dont la réponse n'est pas dans le corpus.",
            f"SUJET VOISIN : {sujet}",
            client=client,
            model=model,
        )
        jeu.append(
            {
                "question": question,
                "reponse": "ABSTENTION_ATTENDUE",
                "passages_attendus": [],
                "type": "sans_reponse",
            }
        )
    return jeu


if __name__ == "__main__" and not openai_configure():
    print("Exemple prêt : configurez OPENAI_API_KEY et OPENAI_MODEL pour générer le jeu.")


## 2. Questions discriminantes

Transformer les voisins du retriever en distracteurs réalistes.

Script correspondant : [`02_questions_discriminantes.py`](examples/02_questions_discriminantes.py)

In [ ]:
# ruff: noqa: F811
"""Générer une question qui sépare un passage de ses voisins proches."""

from __future__ import annotations

from rag_en_pratique.observability import PassageEvaluation, extraire_json
from rag_en_pratique.prompting import generer, openai_configure


def generer_question_discriminante(
    passage_cible: PassageEvaluation,
    index,
    n_distracteurs: int = 2,
    *,
    client=None,
    model: str | None = None,
) -> dict[str, object]:
    voisins = [
        passage
        for passage in index.chercher(passage_cible.texte, k=n_distracteurs + 1)
        if passage.id != passage_cible.id
    ][:n_distracteurs]
    distracteurs = "\n".join(
        f"[Distracteur {numero}] {passage.texte}"
        for numero, passage in enumerate(voisins, start=1)
    )
    sortie = generer(
        "Retourne uniquement un JSON avec les clés question et reponse.",
        (
            f"PASSAGE CIBLE :\n{passage_cible.texte}\n\nDISTRACTEURS :\n{distracteurs}\n\n"
            "Écris une question réaliste que seul le passage cible permet de résoudre, "
            "mais qui partage du vocabulaire avec les distracteurs."
        ),
        client=client,
        model=model,
    )
    resultat = extraire_json(sortie)
    return {
        "question": str(resultat["question"]),
        "reponse": str(resultat["reponse"]),
        "passages_attendus": [passage_cible.id],
        "distracteurs_connus": [passage.id for passage in voisins],
        "type": "discriminante",
    }


if __name__ == "__main__" and not openai_configure():
    print("Exemple prêt : configurez OpenAI et injectez votre index.")


## 3. Filtrage par réalisme

Écarter les questions synthétiques artificielles avant l'évaluation.

Script correspondant : [`03_filtrage_questions.py`](examples/03_filtrage_questions.py)

In [ ]:
# ruff: noqa: F811
"""Filtrer des questions synthétiques selon leur réalisme."""

from __future__ import annotations

from rag_en_pratique.observability import extraire_note
from rag_en_pratique.prompting import generer, openai_configure

INSTRUCTIONS = """Note le réalisme d'une question de recherche immobilière de 1 à 5.
5 = formulation naturelle et besoin plausible ; 1 = détail absurde ou demande artificielle.
Réponds avec une brève explication, puis une ligne « Note : N »."""


def filtrer(
    questions: list[dict[str, object]],
    note_minimale: int = 4,
    *,
    client=None,
    model: str | None = None,
) -> list[dict[str, object]]:
    retenues = []
    for question in questions:
        sortie = generer(
            INSTRUCTIONS,
            f"QUESTION : {question['question']}",
            client=client,
            model=model,
        )
        note = extraire_note(sortie)
        if note >= note_minimale:
            retenues.append({**question, "note_realisme": note, "justification": sortie})
    return retenues


if __name__ == "__main__" and not openai_configure():
    print("Exemple prêt : configurez OPENAI_API_KEY et OPENAI_MODEL.")


## 4. Prompt du juge

Définir le critère, l'échelle et une justification auditable.

Script correspondant : [`04_prompt_juge.py`](examples/04_prompt_juge.py)

In [ ]:
# ruff: noqa: F811
"""Construire le prompt auditable d'un juge de fidélité."""

from rag_en_pratique.observability import PROMPT_JUGE_FAITHFULNESS


def construire_prompt_juge(contexte: str, question: str, reponse: str) -> str:
    return PROMPT_JUGE_FAITHFULNESS.format(
        contexte=contexte,
        question=question,
        reponse=reponse,
    )


if __name__ == "__main__":
    print(construire_prompt_juge("Garantie 24 mois.", "Quelle durée ?", "24 mois."))


## 5. Exécution du juge

Normaliser la note et conserver la sortie brute.

Script correspondant : [`05_execution_juge.py`](examples/05_execution_juge.py)

In [ ]:
# ruff: noqa: F811
"""Exécuter un juge OpenAI et conserver sa justification."""

from __future__ import annotations

import re
from dataclasses import dataclass

from rag_en_pratique.observability import PROMPT_JUGE_FAITHFULNESS
from rag_en_pratique.prompting import generer, openai_configure


@dataclass(frozen=True)
class Verdict:
    note: float
    note_brute: int
    justification: str
    sortie_brute: str


def juger(
    question: str,
    contexte: str,
    reponse: str,
    *,
    client=None,
    model: str | None = None,
) -> Verdict:
    sortie = generer(
        "Respecte exactement la procédure et le format demandés.",
        PROMPT_JUGE_FAITHFULNESS.format(
            contexte=contexte[:3000],
            question=question,
            reponse=reponse,
        ),
        client=client,
        model=model,
    )
    note_match = re.search(r"NOTE\s*:\s*([1-5])", sortie, re.IGNORECASE)
    if not note_match:
        raise ValueError("Le juge n'a pas produit de note valide")
    note_brute = int(note_match.group(1))
    justification = re.search(
        r"JUSTIFICATION\s*:\s*(.+?)(?=NOTE\s*:)",
        sortie,
        re.DOTALL | re.IGNORECASE,
    )
    return Verdict(
        note=(note_brute - 1) / 4,
        note_brute=note_brute,
        justification=justification.group(1).strip() if justification else sortie[:300],
        sortie_brute=sortie,
    )


if __name__ == "__main__" and not openai_configure():
    print("Exemple prêt : configurez OPENAI_API_KEY et OPENAI_MODEL.")


## 6. Deux outils complémentaires

Séparer campagne exploratoire et seuil bloquant de non-régression.

Script correspondant : [`06_deux_outils.py`](examples/06_deux_outils.py)

In [ ]:
# ruff: noqa: F811
"""Comparer campagne de mesure et test de non-régression."""

from __future__ import annotations

import statistics
from collections.abc import Callable
from dataclasses import dataclass


@dataclass(frozen=True)
class CasDeTest:
    question: str
    reponse: str
    contexte: str


def evaluer_campagne(
    jeu: list[CasDeTest],
    metriques: dict[str, Callable[[CasDeTest], float]],
) -> dict[str, object]:
    details = []
    for cas in jeu:
        scores = {nom: mesure(cas) for nom, mesure in metriques.items()}
        details.append({"question": cas.question, **scores})
    moyennes = {
        nom: statistics.mean(ligne[nom] for ligne in details)
        for nom in metriques
        if details
    }
    return {"moyennes": moyennes, "details": details}


def verifier_non_regression(
    cas: CasDeTest,
    mesurer: Callable[[CasDeTest], float],
    seuil: float,
) -> float:
    score = mesurer(cas)
    if score < seuil:
        raise AssertionError(f"Score {score:.3f} inférieur au seuil {seuil:.3f}")
    return score


if __name__ == "__main__":
    exemple = CasDeTest("Durée ?", "24 mois [doc_1]", "Garantie 24 mois")
    mesure_locale = lambda cas: float("24 mois" in cas.reponse and "24 mois" in cas.contexte)
    print(evaluer_campagne([exemple], {"fidelite": mesure_locale}))
    print(verifier_non_regression(exemple, mesure_locale, seuil=0.8))


## 7. Matrice d'attribution

Localiser les défauts du retriever, du générateur ou de l'ancrage.

Script correspondant : [`07_matrice_attribution.py`](examples/07_matrice_attribution.py)

In [ ]:
# ruff: noqa: F811
"""Attribuer les défaillances au retriever, au générateur ou à l'ancrage."""

from __future__ import annotations

from collections import Counter
from collections.abc import Callable
from typing import Any


def _identifiant(source: Any) -> str:
    if hasattr(source, "id"):
        return str(source.id)
    return str(source.metadata.get("id", source.metadata.get("source", "")))


def attribuer_defaillances(
    jeu: list[dict[str, object]],
    systeme,
    juger_exactitude: Callable[[str, str, str], bool],
) -> dict[str, object]:
    compteurs: Counter[str] = Counter()
    exemples = {nom: [] for nom in ("generateur", "retriever", "ancrage", "nominal")}
    for entree in jeu:
        question = str(entree["question"])
        sortie = systeme.repondre(question)
        obtenus = {_identifiant(source) for source in sortie["sources"]}
        attendus = {str(item) for item in entree["passages_attendus"]}
        retrieval_ok = bool(attendus & obtenus) if attendus else not obtenus
        reponse_ok = juger_exactitude(question, sortie["reponse"], str(entree["reponse"]))
        if retrieval_ok and reponse_ok:
            categorie = "nominal"
        elif retrieval_ok:
            categorie = "generateur"
        elif not reponse_ok:
            categorie = "retriever"
        else:
            categorie = "ancrage"
        compteurs[categorie] += 1
        exemples[categorie].append(question)
    total = sum(compteurs.values())
    repartition = {
        nom: round(compteurs[nom] / total, 3) if total else 0.0 for nom in exemples
    }
    priorite = "retriever" if compteurs["retriever"] >= compteurs["generateur"] else "generateur"
    return {
        "repartition": repartition,
        "priorite": priorite,
        "alerte_ancrage": repartition["ancrage"] > 0.05,
        "exemples": exemples,
    }


if __name__ == "__main__":
    print("Injectez un système RAG et une fonction d'exactitude dans attribuer_defaillances().")


## 8. Substitution de contexte

Vérifier que la réponse change quand le contexte utile disparaît.

Script correspondant : [`08_substitution_contexte.py`](examples/08_substitution_contexte.py)

In [ ]:
# ruff: noqa: F811
"""Tester si un système RAG dépend réellement du contexte fourni."""

from __future__ import annotations

import random
import re


def similarite_lexicale(premier: str, second: str) -> float:
    a = set(re.findall(r"\w+", premier.lower()))
    b = set(re.findall(r"\w+", second.lower()))
    return len(a & b) / len(a | b) if a or b else 1.0


def est_une_abstention(texte: str) -> bool:
    normalise = texte.lower()
    marqueurs = (
        "ne permettent pas de répondre",
        "information absente",
        "je ne peux pas répondre",
        "abstention",
    )
    return any(marqueur in normalise for marqueur in marqueurs)


def tester_ancrage(
    jeu: list[dict[str, object]],
    systeme,
    corpus_etranger: list[object],
    seuil_similarite: float = 0.75,
    *,
    taille_contexte: int = 5,
    graine: int = 42,
) -> dict[str, object]:
    if not jeu:
        raise ValueError("Le jeu d'évaluation ne peut pas être vide")
    if not corpus_etranger:
        raise ValueError("Le corpus étranger ne peut pas être vide")
    rng = random.Random(graine)
    identiques = abstentions = 0
    for entree in jeu:
        question = str(entree["question"])
        reponse_reelle = systeme.repondre(question)["reponse"]
        contexte = rng.sample(corpus_etranger, min(taille_contexte, len(corpus_etranger)))
        reponse_substituee = systeme.generer_avec(question, contexte=contexte)
        if est_une_abstention(reponse_substituee):
            abstentions += 1
        elif similarite_lexicale(reponse_reelle, reponse_substituee) > seuil_similarite:
            identiques += 1
    total = len(jeu)
    taux_abstention = abstentions / total
    return {
        "taux_abstention": round(taux_abstention, 3),
        "taux_reponse_de_memoire": round(identiques / total, 3),
        "ancrage_suffisant": taux_abstention > 0.85,
    }


if __name__ == "__main__":
    print(round(similarite_lexicale("garantie 24 mois", "garantie de 24 mois"), 3))


## 9. Percentiles de latence

Suivre p50, p95 et p99 sur une fenêtre bornée.

Script correspondant : [`09_percentiles.py`](examples/09_percentiles.py)

In [ ]:
# ruff: noqa: F811
"""Suivre des percentiles glissants de latence, étape par étape."""

from __future__ import annotations

import math
import statistics
import time
from collections import deque

ETAPES = ("vectorisation", "recherche", "reclassement", "generation", "total")


def percentile(valeurs: list[float], quantile: float) -> float:
    if not valeurs:
        raise ValueError("Aucune mesure")
    if not 0 < quantile <= 1:
        raise ValueError("Le quantile doit appartenir à ]0, 1]")
    triees = sorted(valeurs)
    return triees[math.ceil(quantile * len(triees)) - 1]


class SuiviLatence:
    def __init__(self, fenetre: int = 1000, minimum: int = 20) -> None:
        if fenetre <= 0 or minimum <= 0:
            raise ValueError("fenetre et minimum doivent être positifs")
        self.minimum = minimum
        self.mesures = {etape: deque(maxlen=fenetre) for etape in ETAPES}

    def chronometrer(self, etape: str):
        if etape not in self.mesures:
            raise KeyError(f"Étape inconnue : {etape}")
        return _Chrono(self.mesures[etape])

    def enregistrer(self, etape: str, millisecondes: float) -> None:
        if millisecondes < 0:
            raise ValueError("Une latence ne peut pas être négative")
        self.mesures[etape].append(float(millisecondes))

    def percentiles(self) -> dict[str, dict[str, float | int]]:
        rapport = {}
        for etape, valeurs in self.mesures.items():
            if len(valeurs) < self.minimum:
                continue
            liste = list(valeurs)
            rapport[etape] = {
                "p50": round(statistics.median(liste), 1),
                "p95": round(percentile(liste, 0.95), 1),
                "p99": round(percentile(liste, 0.99), 1),
                "n": len(liste),
            }
        return rapport

    def part_de_chaque_etape(self) -> dict[str, str]:
        rapport = self.percentiles()
        total = rapport.get("total", {}).get("p50")
        if not total:
            return {}
        return {
            etape: f"{float(valeurs['p50']) / float(total) * 100:.0f} %"
            for etape, valeurs in rapport.items()
            if etape != "total"
        }


class _Chrono:
    def __init__(self, cible: deque) -> None:
        self.cible = cible
        self.debut = 0.0

    def __enter__(self):
        self.debut = time.perf_counter()
        return self

    def __exit__(self, *_: object) -> None:
        self.cible.append((time.perf_counter() - self.debut) * 1000)


if __name__ == "__main__":
    suivi = SuiviLatence(minimum=5)
    for valeur in [10, 12, 14, 18, 40]:
        suivi.enregistrer("recherche", valeur)
    print(suivi.percentiles())


## 10. Détection de dérive

Combiner significativité statistique et ampleur pratique.

Script correspondant : [`10_detection_drift.py`](examples/10_detection_drift.py)

In [ ]:
# ruff: noqa: F811
"""Détecter une dérive statistiquement significative et matériellement importante."""

from __future__ import annotations

import numpy as np
from scipy.stats import ks_2samp


def detecter_derive(
    reference: dict[str, list[float]],
    actuel: dict[str, list[float]],
    seuil_p: float = 0.05,
    seuil_ecart: float = 0.05,
    *,
    plus_grand_est_meilleur: dict[str, bool] | None = None,
) -> list[dict[str, object]]:
    directions = plus_grand_est_meilleur or {}
    alertes = []
    for metrique, valeurs_actuelles in actuel.items():
        valeurs_ref = reference.get(metrique)
        if not valeurs_ref or not valeurs_actuelles:
            continue
        _, p_valeur = ks_2samp(valeurs_ref, valeurs_actuelles)
        moyenne_ref = float(np.mean(valeurs_ref))
        moyenne_actuelle = float(np.mean(valeurs_actuelles))
        ecart = moyenne_actuelle - moyenne_ref
        if p_valeur >= seuil_p or abs(ecart) <= seuil_ecart:
            continue
        hausse_souhaitable = directions.get(metrique, True)
        degradation = ecart < 0 if hausse_souhaitable else ecart > 0
        alertes.append(
            {
                "metrique": metrique,
                "reference": round(moyenne_ref, 3),
                "actuel": round(moyenne_actuelle, 3),
                "ecart_points": round(ecart * 100, 1),
                "p_valeur": round(float(p_valeur), 4),
                "sens": "degradation" if degradation else "amelioration",
            }
        )
    return alertes


if __name__ == "__main__":
    rng = np.random.default_rng(42)
    reference = {"faithfulness": rng.normal(0.90, 0.02, 200).tolist()}
    actuel = {"faithfulness": rng.normal(0.75, 0.02, 200).tolist()}
    print(detecter_derive(reference, actuel))


## Bilan

L'observabilité utile relie chaque alerte à des exemples inspectables. Conservez les traces nécessaires, mesurez les distributions plutôt que les seules moyennes, et calibrez régulièrement les juges automatiques sur des humains.